# A Walkthrough of Google's PageRank Algorithm

So here is my walk through of Google's original PageRank algorithm step by step

The core idea behind PageRank is pretty simple: "A webpage is important if important pages link to it". This definition turns out to have a precise linear algebra formulation as the PageRank vector is the dominant eigenvector of a particular matrix. Everything that follows is about constructing that matrix correctly and extracting that eigenvector.

I'll work through:
1. Building a transition matrix from a directed graph of web links
2. Understanding why PageRank is an eigenvalue problem
3. Diagnosing and fixing a broken matrix (dangling nodes)
4. Computing PageRank via eigendecomposition and the power method
5. Adding Google's teleportation mechanism (the damping factor)
6. Using PageRank to rank search results via a term-document matrix

In [1]:
import numpy as np

# The 12 pages in "Google Network #10"
pages = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']
num_pages = len(pages)

page_names = {
    'A': 'Trees For All',      'B': 'The Weeping Willow',
    'C': 'Leaf Fall',          'D': 'Shady Place',
    'E': 'The Prickly Pine',   'F': 'Good Wood',
    'G': 'Lumber Jack Man',    'H': 'Timber!',
    'I': 'Fine Grains',        'J': 'The Warped Board',
    'K': 'Evergreen Plants',   'L': 'Sappy Planks',
}

# Directed links: from_page -> [to_pages]
adjacency = {
    'A': ['B'],           'B': ['C'],
    'C': ['D', 'G'],      'D': ['A', 'H'],
    'E': ['A', 'F'],      'F': ['B', 'J', 'K'],
    'G': ['F', 'H'],      'H': ['L'],
    'I': ['E', 'J'],      'J': [],              # dangling node!
    'K': ['G', 'J'],      'L': ['D', 'K'],
}

# Keywords for each page (used later for search)
page_keywords = {
    'A': ['Ash', 'Butternut', 'Cherry', 'Elm', 'Katsura', 'Magnolia', 'Teak', 'Ginkgo'],
    'B': ['Butternut', 'Fir', 'Hickory', 'Magnolia', 'Pine', 'Willow', 'Redwood', 'Sassafras'],
    'C': ['Ash', 'Elm', 'Hickory', 'Katsura', 'Oak', 'Ginkgo', 'Redwood'],
    'D': ['Butternut', 'Cherry', 'Fir', 'Spruce', 'Teak', 'Aspen', 'Sassafras'],
    'E': ['Cherry', 'Hickory', 'Oak', 'Pine', 'Willow', 'Redwood'],
    'F': ['Ash', 'Fir', 'Magnolia', 'Spruce', 'Ginkgo', 'Redwood', 'Aspen', 'Sassafras'],
    'G': ['Ash', 'Butternut', 'Oak', 'Spruce', 'Ginkgo', 'Redwood'],
    'H': ['Ash', 'Cherry', 'Hickory', 'Willow', 'Redwood', 'Aspen'],
    'I': ['Elm', 'Fir', 'Katsura', 'Magnolia', 'Pine', 'Spruce', 'Sassafras'],
    'J': ['Magnolia', 'Oak', 'Willow', 'Redwood', 'Aspen', 'Sassafras'],
    'K': ['Cherry', 'Elm', 'Fir', 'Hickory', 'Teak', 'Ginkgo', 'Redwood', 'Sassafras'],
    'L': ['Butternut', 'Elm', 'Katsura', 'Oak', 'Pine', 'Spruce', 'Teak', 'Ginkgo',
          'Aspen', 'Sassafras'],
}

all_keywords = sorted(set(kw for kws in page_keywords.values() for kw in kws))

def page_index(label):
    return pages.index(label)

print(f"Network: {num_pages} pages, {len(all_keywords)} unique keywords")
print(f"Pages: {pages}")
print(f"Keywords: {all_keywords}")

Network: 12 pages, 17 unique keywords
Pages: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']
Keywords: ['Ash', 'Aspen', 'Butternut', 'Cherry', 'Elm', 'Fir', 'Ginkgo', 'Hickory', 'Katsura', 'Magnolia', 'Oak', 'Pine', 'Redwood', 'Sassafras', 'Spruce', 'Teak', 'Willow']


## Step 1: From Network to Google Matrix

I need to turn this directed graph into a matrix that captures the probability of a random surfer moving from one page to another. The key idea is:

> If page $i$ has $k$ outgoing links, then the surfer on page $i$ clicks each link with probability $\frac{1}{k}$.

So I construct the **Google Matrix** $G$ where:

$$G_{ij} = \begin{cases} \frac{1}{\text{out-degree}(i)} & \text{if page } i \text{ links to page } j \\ 0 & \text{otherwise} \end{cases}$$

Each row of $G$ represents a probability distribution over where the surfer goes next. That's why **every row must sum to 1** — the surfer *has* to go somewhere. A matrix where every row sums to 1 is called **row stochastic**, and this property is what makes it a valid transition matrix for a Markov chain.

If a row doesn't sum to 1, it means probability is "leaking" — we're not accounting for all possible transitions. That would break the entire framework.

In [2]:
G_raw = np.zeros((num_pages, num_pages))

for from_page, to_pages in adjacency.items():
    i = page_index(from_page)
    num_links = len(to_pages)
    if num_links > 0:
        for to_page in to_pages:
            j = page_index(to_page)
            G_raw[i][j] = 1.0 / num_links

row_sums = G_raw.sum(axis=1)

print("Raw Google Matrix G:")
print(np.array2string(G_raw, precision=4, suppress_small=True))
print(f"\nRow sums: {row_sums}")
print(f"\nRow for J (index {page_index('J')}) sums to {row_sums[page_index('J')]:.1f} — this is the problem.")

Raw Google Matrix G:
[[0.     1.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.    ]
 [0.     0.     1.     0.     0.     0.     0.     0.     0.     0.
  0.     0.    ]
 [0.     0.     0.     0.5    0.     0.     0.5    0.     0.     0.
  0.     0.    ]
 [0.5    0.     0.     0.     0.     0.     0.     0.5    0.     0.
  0.     0.    ]
 [0.5    0.     0.     0.     0.     0.5    0.     0.     0.     0.
  0.     0.    ]
 [0.     0.3333 0.     0.     0.     0.     0.     0.     0.     0.3333
  0.3333 0.    ]
 [0.     0.     0.     0.     0.     0.5    0.     0.5    0.     0.
  0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     1.    ]
 [0.     0.     0.     0.     0.5    0.     0.     0.     0.     0.5
  0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.5    0.     0.     0.5
  0.     0.    ]
 [0.     0.     0.     0.5    0.     0

## Steps 2–3: The Eigenvalue Problem — and Why the Raw Matrix Fails

Here's the core mathematical insight that makes PageRank work. I'm looking for a **steady-state distribution** $\pi$ — a vector that doesn't change when I apply the transition matrix:

$$\pi^T G = \pi^T$$

This says: if the surfer population is distributed according to $\pi$, then after one more round of clicking, the distribution is *still* $\pi$. The system has reached equilibrium.

Now I can rewrite this. Transposing both sides:

$$G^T \pi = 1 \cdot \pi$$

This is literally **an eigenvalue equation**: $\pi$ is the eigenvector of $G^T$ with eigenvalue $\lambda = 1$. The PageRank vector *is* the eigenvector for the dominant eigenvalue.

But the catch is: does the raw matrix even *have* eigenvalue 1? For a row-stochastic matrix, the Perron-Frobenius theorem guarantees that the dominant eigenvalue is exactly 1. But our raw matrix **is not stochastic** as page J's row is all zeros, so that row sums to 0 instead of 1.

I expect the dominant eigenvalue of the raw $G^T$ to be **less than 1**. Let me check.

In [3]:
eigenvalues_raw, eigenvectors_raw = np.linalg.eig(G_raw.T)

print("Eigenvalues of raw G^T:")
for idx, val in enumerate(eigenvalues_raw):
    marker = " <-- closest to 1" if idx == np.argmin(np.abs(eigenvalues_raw - 1.0)) else ""
    print(f"  lambda_{idx} = {val:.6f}{marker}")

dominant_idx = np.argmax(np.abs(eigenvalues_raw))
dominant_eigenvalue = np.abs(eigenvalues_raw[dominant_idx])
print(f"\nDominant eigenvalue magnitude: {dominant_eigenvalue:.6f}")
print(f"Is it equal to 1? {'Yes' if abs(dominant_eigenvalue - 1.0) < 1e-10 else 'NO — it is less than 1!'}")

closest_idx = np.argmin(np.abs(eigenvalues_raw - 1.0))
closest_eigenvector = eigenvectors_raw[:, closest_idx].real
print(f"\nClosest eigenvalue to 1: {eigenvalues_raw[closest_idx]:.10f}")
print(f"Its eigenvector: {closest_eigenvector}")

has_mixed_signs = not (np.all(closest_eigenvector >= -1e-12) or np.all(closest_eigenvector <= 1e-12))
if has_mixed_signs:
    print("\nThis eigenvector has MIXED SIGNS — it cannot represent a probability distribution.")
    print("The raw matrix is broken. We need to fix the dangling node.")

Eigenvalues of raw G^T:
  lambda_0 = 0.000000+0.000000j
  lambda_1 = -0.139267+0.843446j
  lambda_2 = -0.139267-0.843446j
  lambda_3 = -0.613578+0.161198j
  lambda_4 = -0.613578-0.161198j
  lambda_5 = 0.927933+0.000000j <-- closest to 1
  lambda_6 = 0.084565+0.423859j
  lambda_7 = 0.084565-0.423859j
  lambda_8 = 0.408626+0.000000j
  lambda_9 = 0.000000+0.000000j
  lambda_10 = 0.000000+0.000000j
  lambda_11 = 0.000000+0.000000j

Dominant eigenvalue magnitude: 0.927933
Is it equal to 1? NO — it is less than 1!

Closest eigenvalue to 1: 0.9279332151+0.0000000000j
Its eigenvector: [0.21519369 0.2955103  0.31846074 0.39937074 0.         0.1770604
 0.32860045 0.39225408 0.         0.22060751 0.2913778  0.42271801]


## Step 4: Identifying and Fixing the Dangling Node

The problem is clear: **page J has zero outgoing links**. Its row in $G$ is all zeros, which means:

- The row sum is 0, not 1 — $G$ is not stochastic
- A random surfer who lands on J is "stuck" with no links to follow
- The matrix loses the mathematical properties we need (Perron-Frobenius doesn't apply)

The fix is both practical and elegant: **replace J's zero row with a uniform distribution** $\frac{1}{n}$ over all pages. The interpretation is natural — when a surfer reaches a dead end, they pick a random page to visit (like typing a random URL or using a bookmark).

$$G_{J,:} = \left[\frac{1}{12}, \frac{1}{12}, \ldots, \frac{1}{12}\right]$$

After this fix, every row sums to exactly 1, making $G$ a proper row-stochastic matrix.

In [4]:
G = G_raw.copy()
j_idx = page_index('J')
G[j_idx, :] = 1.0 / num_pages

row_sums_fixed = G.sum(axis=1)

print(f"Page J (index {j_idx}): replaced zero row with uniform 1/{num_pages} = {1.0/num_pages:.6f}")
print(f"\nFixed Google Matrix G:")
print(np.array2string(G, precision=4, suppress_small=True))
print(f"\nRow sums: {row_sums_fixed}")
print(f"All rows sum to 1: {np.allclose(row_sums_fixed, 1.0)}")

Page J (index 9): replaced zero row with uniform 1/12 = 0.083333

Fixed Google Matrix G:
[[0.     1.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.    ]
 [0.     0.     1.     0.     0.     0.     0.     0.     0.     0.
  0.     0.    ]
 [0.     0.     0.     0.5    0.     0.     0.5    0.     0.     0.
  0.     0.    ]
 [0.5    0.     0.     0.     0.     0.     0.     0.5    0.     0.
  0.     0.    ]
 [0.5    0.     0.     0.     0.     0.5    0.     0.     0.     0.
  0.     0.    ]
 [0.     0.3333 0.     0.     0.     0.     0.     0.     0.     0.3333
  0.3333 0.    ]
 [0.     0.     0.     0.     0.     0.5    0.     0.5    0.     0.
  0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     1.    ]
 [0.     0.     0.     0.     0.5    0.     0.     0.     0.     0.5
  0.     0.    ]
 [0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833
  0.0833 0.0833]
 [0.     0.     0.     0.     0.     0.     0.5    0

## Steps 5–6: Eigendecomposition of the Fixed Matrix

Now that $G$ is properly stochastic, I expect the Perron-Frobenius theorem to deliver: the dominant eigenvalue should be exactly $\lambda = 1$, and its eigenvector should have all non-negative entries.

I'll compute the eigendecomposition of $G^T$, find the eigenvector for $\lambda = 1$, and normalize it so its entries sum to 1. That normalized vector **is the PageRank** — each entry $\pi_i$ represents the long-run fraction of time a random surfer spends on page $i$.

In [5]:
eigenvalues_fixed, eigenvectors_fixed = np.linalg.eig(G.T)

print("Eigenvalues of fixed G^T:")
for idx, val in enumerate(eigenvalues_fixed):
    marker = " <-- eigenvalue 1!" if abs(val - 1.0) < 1e-10 else ""
    print(f"  lambda_{idx} = {val:.6f}{marker}")

eig1_idx = np.argmin(np.abs(eigenvalues_fixed - 1.0))
pi_eigen = eigenvectors_fixed[:, eig1_idx].real

if np.all(pi_eigen <= 1e-12):
    pi_eigen = -pi_eigen

pi_eigen = pi_eigen / pi_eigen.sum()

print(f"\nNormalized PageRank vector (eigendecomposition of fixed G):")
for i, label in enumerate(pages):
    print(f"  Page {label} ({page_names[label]:<25}): {pi_eigen[i]:.8f}")
print(f"\nSum: {pi_eigen.sum():.10f}")

Eigenvalues of fixed G^T:
  lambda_0 = 1.000000+0.000000j <-- eigenvalue 1!
  lambda_1 = -0.137303+0.842133j
  lambda_2 = -0.137303-0.842133j
  lambda_3 = -0.600523+0.159728j
  lambda_4 = -0.600523-0.159728j
  lambda_5 = -0.373605+0.000000j
  lambda_6 = 0.003537+0.461959j
  lambda_7 = 0.003537-0.461959j
  lambda_8 = 0.303288+0.171572j
  lambda_9 = 0.303288-0.171572j
  lambda_10 = 0.318940+0.000000j
  lambda_11 = -0.000000+0.000000j

Normalized PageRank vector (eigendecomposition of fixed G):
  Page A (Trees For All            ): 0.07252018
  Page B (The Weeping Willow       ): 0.10005767
  Page C (Leaf Fall                ): 0.10640138
  Page D (Shady Place              ): 0.12283737
  Page E (The Prickly Pine         ): 0.00951557
  Page F (Good Wood                ): 0.06358131
  Page G (Lumber Jack Man          ): 0.10495963
  Page H (Timber!                  ): 0.12024221
  Page I (Fine Grains              ): 0.00634371
  Page J (The Warped Board         ): 0.07612457
  Page K (Eve

## Step 7: Ranking Pages by Importance

Now I simply sort pages by their PageRank values. Higher PageRank = more important = the random surfer spends more time there.

In [6]:
ranking_G = sorted(
    [(pages[i], pi_eigen[i]) for i in range(num_pages)],
    key=lambda x: x[1],
    reverse=True
)

print(f"{'Rank':<6} {'Page':<6} {'Name':<25} {'PageRank':<12}")
print(f"{'-'*6} {'-'*6} {'-'*25} {'-'*12}")
for rank, (label, value) in enumerate(ranking_G, 1):
    print(f"{rank:<6} {label:<6} {page_names[label]:<25} {value:.8f}")

Rank   Page   Name                      PageRank    
------ ------ ------------------------- ------------
1      L      Sappy Planks              0.12658593
2      D      Shady Place               0.12283737
3      H      Timber!                   0.12024221
4      C      Leaf Fall                 0.10640138
5      G      Lumber Jack Man           0.10495963
6      B      The Weeping Willow        0.10005767
7      K      Evergreen Plants          0.09083045
8      J      The Warped Board          0.07612457
9      A      Trees For All             0.07252018
10     F      Good Wood                 0.06358131
11     E      The Prickly Pine          0.00951557
12     I      Fine Grains               0.00634371


## Step 8: The Power Method — and Why It Converges to the Dominant Eigenvector

Eigendecomposition works on our small $12 \times 12$ matrix, but the real web has billions of pages. Google can't compute a full eigendecomposition of a billion-dimensional matrix. Instead, they use the **power method** — an iterative algorithm that's conceptually simple and scales beautifully.

The algorithm:
1. Start with any distribution $\pi_0$ (I'll use uniform: $\pi_0 = [\frac{1}{n}, \ldots, \frac{1}{n}]$)
2. Iterate: $\pi_{k+1} = \pi_k \cdot G$
3. Stop when $\|\pi_{k+1} - \pi_k\|$ is small enough

**Why does this work?** Any starting vector $\pi_0$ can be decomposed into the eigenvectors of $G^T$:

$$\pi_0 = c_1 v_1 + c_2 v_2 + \cdots + c_n v_n$$

where $v_1$ is the dominant eigenvector (eigenvalue $\lambda_1 = 1$) and $|\lambda_2| \leq |\lambda_3| \leq \cdots < 1$. After $k$ multiplications by $G$:

$$\pi_k = c_1 \lambda_1^k v_1 + c_2 \lambda_2^k v_2 + \cdots = c_1 v_1 + c_2 \lambda_2^k v_2 + \cdots$$

Since $|\lambda_i| < 1$ for $i \geq 2$, those terms $\lambda_i^k \to 0$ as $k \to \infty$. Each iteration, the non-dominant components **shrink by a factor of** $|\lambda_2 / \lambda_1|$. After enough iterations, only the dominant eigenvector $v_1$ survives — which is exactly the PageRank vector.

The convergence rate depends on the **spectral gap** $|\lambda_1| - |\lambda_2|$. A larger gap means faster convergence.

In [7]:
pi_power = np.ones(num_pages) / num_pages
tolerance = 1e-10
max_iterations = 1000

for iteration in range(1, max_iterations + 1):
    pi_new = pi_power @ G
    pi_new = pi_new / pi_new.sum()
    change = np.sum(np.abs(pi_new - pi_power))
    if change < tolerance:
        pi_power = pi_new
        print(f"Converged after {iteration} iterations (L1 change: {change:.2e})")
        break
    pi_power = pi_new
else:
    print(f"WARNING: Did not converge after {max_iterations} iterations.")

print(f"\nPower method result:")
for i, label in enumerate(pages):
    print(f"  Page {label}: {pi_power[i]:.8f}")

diff = np.sum(np.abs(pi_power - pi_eigen))
print(f"\nL1 difference vs eigendecomposition: {diff:.2e}")
print(f"Methods agree: {diff < 1e-6}")

Converged after 134 iterations (L1 change: 9.17e-11)

Power method result:
  Page A: 0.07252018
  Page B: 0.10005767
  Page C: 0.10640138
  Page D: 0.12283737
  Page E: 0.00951557
  Page F: 0.06358131
  Page G: 0.10495963
  Page H: 0.12024221
  Page I: 0.00634371
  Page J: 0.07612457
  Page K: 0.09083045
  Page L: 0.12658593

L1 difference vs eigendecomposition: 5.90e-11
Methods agree: True


## Step 9: The Adjusted Google Matrix and Teleportation

The fixed matrix $G$ works, but Google goes one step further with the **adjusted Google Matrix** $\tilde{G}$:

$$\tilde{G} = \alpha G + (1 - \alpha) \frac{1}{n} \mathbf{e}\mathbf{e}^T$$

where $\alpha = 0.85$ is the **damping factor** and $\mathbf{e}$ is the vector of all ones.

What does this mean physically? At each step, the random surfer:
- With probability $\alpha = 0.85$: follows a link on the current page (uses $G$)
- With probability $1 - \alpha = 0.15$: gets "bored" and **teleports** to a completely random page (uses $\frac{1}{n}\mathbf{e}\mathbf{e}^T$)

This teleportation mechanism does something crucial mathematically: it makes **every entry of $\tilde{G}$ strictly positive** (since the $\frac{1}{n}\mathbf{e}\mathbf{e}^T$ term adds $\frac{0.15}{12} \approx 0.0125$ to every entry). A matrix with all positive entries is called **primitive**, and the Perron-Frobenius theorem for primitive matrices guarantees:

1. There is a **unique** dominant eigenvalue of exactly 1
2. The corresponding eigenvector has all **strictly positive** entries
3. The power method is **guaranteed to converge** regardless of the starting vector

Without teleportation, the matrix might have multiple eigenvalues of magnitude 1 (e.g., if the graph has disconnected components), which would mean the power method could cycle or converge to different vectors depending on the starting point. Teleportation eliminates this possibility.

In [8]:
alpha = 0.85
random_jump_matrix = np.ones((num_pages, num_pages)) / num_pages
G_tilde = alpha * G + (1 - alpha) * random_jump_matrix

row_sums_tilde = G_tilde.sum(axis=1)

print(f"G_tilde = {alpha} * G + {1-alpha} * (1/{num_pages}) * ones({num_pages},{num_pages})")
print(f"\nAdjusted Google Matrix G_tilde:")
print(np.array2string(G_tilde, precision=4, suppress_small=True))
print(f"\nRow sums: {row_sums_tilde}")
print(f"All positive entries: {np.all(G_tilde > 0)}")
print(f"Smallest entry: {G_tilde.min():.6f}")

G_tilde = 0.85 * G + 0.15000000000000002 * (1/12) * ones(12,12)

Adjusted Google Matrix G_tilde:
[[0.0125 0.8625 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125
  0.0125 0.0125]
 [0.0125 0.0125 0.8625 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125
  0.0125 0.0125]
 [0.0125 0.0125 0.0125 0.4375 0.0125 0.0125 0.4375 0.0125 0.0125 0.0125
  0.0125 0.0125]
 [0.4375 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.4375 0.0125 0.0125
  0.0125 0.0125]
 [0.4375 0.0125 0.0125 0.0125 0.0125 0.4375 0.0125 0.0125 0.0125 0.0125
  0.0125 0.0125]
 [0.0125 0.2958 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.2958
  0.2958 0.0125]
 [0.0125 0.0125 0.0125 0.0125 0.0125 0.4375 0.0125 0.4375 0.0125 0.0125
  0.0125 0.0125]
 [0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125 0.0125
  0.0125 0.8625]
 [0.0125 0.0125 0.0125 0.0125 0.4375 0.0125 0.0125 0.0125 0.0125 0.4375
  0.0125 0.0125]
 [0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833 0.0833
  0.0833 0.0833]
 [0.0125 0.01

## Step 10: PageRank from $\tilde{G}$

I'll compute PageRank from $\tilde{G}$ using both methods — eigendecomposition and the power method — to confirm they agree, just as they did for the fixed $G$. These are **two paths to the same answer**: one solves the eigenvalue equation directly, the other iterates until convergence. The fact that they match validates both the math and the implementation.

In [9]:
# Eigendecomposition
eigenvalues_tilde, eigenvectors_tilde = np.linalg.eig(G_tilde.T)

print("Eigenvalues of G_tilde^T:")
for idx, val in enumerate(eigenvalues_tilde):
    marker = " <-- eigenvalue 1!" if abs(val - 1.0) < 1e-10 else ""
    print(f"  lambda_{idx} = {val:.6f}{marker}")

eig1_idx_tilde = np.argmin(np.abs(eigenvalues_tilde - 1.0))
pi_tilde_eigen = eigenvectors_tilde[:, eig1_idx_tilde].real
if np.all(pi_tilde_eigen <= 1e-12):
    pi_tilde_eigen = -pi_tilde_eigen
pi_tilde_eigen = pi_tilde_eigen / pi_tilde_eigen.sum()

print(f"\nPageRank from eigendecomposition of G_tilde:")
for i, label in enumerate(pages):
    print(f"  Page {label}: {pi_tilde_eigen[i]:.8f}")

# Power method
pi_tilde_power = np.ones(num_pages) / num_pages
for iteration in range(1, max_iterations + 1):
    pi_new = pi_tilde_power @ G_tilde
    pi_new = pi_new / pi_new.sum()
    change = np.sum(np.abs(pi_new - pi_tilde_power))
    if change < tolerance:
        pi_tilde_power = pi_new
        print(f"\nPower method converged after {iteration} iterations (L1 change: {change:.2e})")
        break
    pi_tilde_power = pi_new

diff_tilde = np.sum(np.abs(pi_tilde_power - pi_tilde_eigen))
print(f"L1 difference (eigen vs power): {diff_tilde:.2e}")
print(f"Methods agree: {diff_tilde < 1e-6}")

Eigenvalues of G_tilde^T:
  lambda_0 = 1.000000+0.000000j <-- eigenvalue 1!
  lambda_1 = -0.116708+0.715813j
  lambda_2 = -0.116708-0.715813j
  lambda_3 = -0.510444+0.135768j
  lambda_4 = -0.510444-0.135768j
  lambda_5 = 0.003006+0.392665j
  lambda_6 = 0.003006-0.392665j
  lambda_7 = -0.317564+0.000000j
  lambda_8 = 0.257795+0.145836j
  lambda_9 = 0.257795-0.145836j
  lambda_10 = 0.271099+0.000000j
  lambda_11 = -0.000000+0.000000j

PageRank from eigendecomposition of G_tilde:
  Page A: 0.07642904
  Page B: 0.10374151
  Page C: 0.10655497
  Page D: 0.11041456
  Page E: 0.02618392
  Page F: 0.07200759
  Page G: 0.10001115
  Page H: 0.10780561
  Page I: 0.01837468
  Page J: 0.08293668
  Page K: 0.08553085
  Page L: 0.11000945

Power method converged after 66 iterations (L1 change: 9.69e-11)
L1 difference (eigen vs power): 5.22e-11
Methods agree: True


## Step 11: Comparing $G$ vs $\tilde{G}$ Rankings

How much does the 15% teleportation actually change the rankings? I expect some shuffling but not a dramatic reordering — the link structure still dominates (it's 85% of the matrix). The teleportation mainly:

- Gives every page a small baseline importance (even pages with no incoming links)
- Slightly compresses the range of PageRank values (top pages lose a bit, bottom pages gain a bit)
- Can cause rank swaps among pages with similar PageRank values

In [10]:
ranking_tilde = sorted(
    [(pages[i], pi_tilde_eigen[i]) for i in range(num_pages)],
    key=lambda x: x[1],
    reverse=True
)

print(f"{'Rank':<6} {'--- Fixed G ---':<32} {'--- Adjusted G_tilde ---':<32}")
print(f"{'-'*6} {'-'*32} {'-'*32}")
for rank in range(num_pages):
    g_label, g_val = ranking_G[rank]
    t_label, t_val = ranking_tilde[rank]
    g_str = f"{g_label} ({page_names[g_label]:<20}) {g_val:.6f}"
    t_str = f"{t_label} ({page_names[t_label]:<20}) {t_val:.6f}"
    marker = " *" if g_label != t_label else ""
    print(f"{rank+1:<6} {g_str:<32} {t_str:<32}{marker}")

g_order = [label for label, _ in ranking_G]
t_order = [label for label, _ in ranking_tilde]
if g_order == t_order:
    print("\nRanking order is IDENTICAL — teleportation didn't change the relative ordering.")
else:
    print("\n* = rank position changed between G and G_tilde")
    for rank in range(num_pages):
        if g_order[rank] != t_order[rank]:
            print(f"  Rank {rank+1}: {g_order[rank]} (G) vs {t_order[rank]} (G_tilde)")

Rank   --- Fixed G ---                  --- Adjusted G_tilde ---        
------ -------------------------------- --------------------------------
1      L (Sappy Planks        ) 0.126586 D (Shady Place         ) 0.110415 *
2      D (Shady Place         ) 0.122837 L (Sappy Planks        ) 0.110009 *
3      H (Timber!             ) 0.120242 H (Timber!             ) 0.107806
4      C (Leaf Fall           ) 0.106401 C (Leaf Fall           ) 0.106555
5      G (Lumber Jack Man     ) 0.104960 B (The Weeping Willow  ) 0.103742 *
6      B (The Weeping Willow  ) 0.100058 G (Lumber Jack Man     ) 0.100011 *
7      K (Evergreen Plants    ) 0.090830 K (Evergreen Plants    ) 0.085531
8      J (The Warped Board    ) 0.076125 J (The Warped Board    ) 0.082937
9      A (Trees For All       ) 0.072520 A (Trees For All       ) 0.076429
10     F (Good Wood           ) 0.063581 F (Good Wood           ) 0.072008
11     E (The Prickly Pine    ) 0.009516 E (The Prickly Pine    ) 0.026184
12     I (Fine Grains

## Step 12: The Term-Document Matrix

Now I shift from ranking pages by structural importance to **answering search queries**. I need to connect keywords to pages, and the standard tool for this is a term-document matrix $T$.

I set $T$ up as a binary incidence matrix with keywords as rows and pages as columns:

$$T_{ij} = \begin{cases} 1 & \text{if keyword } i \text{ appears on page } j \\ 0 & \text{otherwise} \end{cases}$$

This is a simple binary model — a keyword is either present or absent. More sophisticated approaches (like TF-IDF) would weight by frequency, but for our purposes binary works fine. The power of this representation is that search becomes matrix multiplication: I can find all pages matching a query by multiplying a query vector by $T$.

In [11]:
T = np.zeros((len(all_keywords), num_pages), dtype=int)

for j, label in enumerate(pages):
    for keyword in page_keywords[label]:
        i = all_keywords.index(keyword)
        T[i][j] = 1

print(f"Term-Document Matrix T ({len(all_keywords)} keywords x {num_pages} pages):\n")
header = f"{'Keyword':<12}" + "".join(f" {label:>3}" for label in pages)
print(header)
print(f"{'-'*12}" + "".join(f" {'---':>3}" for _ in pages))
for i, keyword in enumerate(all_keywords):
    row = f"{keyword:<12}" + "".join(f" {T[i][j]:>3}" for j in range(num_pages))
    print(row)

Term-Document Matrix T (17 keywords x 12 pages):

Keyword        A   B   C   D   E   F   G   H   I   J   K   L
------------ --- --- --- --- --- --- --- --- --- --- --- ---
Ash            1   0   1   0   0   1   1   1   0   0   0   0
Aspen          0   0   0   1   0   1   0   1   0   1   0   1
Butternut      1   1   0   1   0   0   1   0   0   0   0   1
Cherry         1   0   0   1   1   0   0   1   0   0   1   0
Elm            1   0   1   0   0   0   0   0   1   0   1   1
Fir            0   1   0   1   0   1   0   0   1   0   1   0
Ginkgo         1   0   1   0   0   1   1   0   0   0   1   1
Hickory        0   1   1   0   1   0   0   1   0   0   1   0
Katsura        1   0   1   0   0   0   0   0   1   0   0   1
Magnolia       1   1   0   0   0   1   0   0   1   1   0   0
Oak            0   0   1   0   1   0   1   0   0   1   0   1
Pine           0   1   0   0   1   0   0   0   1   0   0   1
Redwood        0   1   1   0   1   1   1   1   0   1   1   0
Sassafras      0   1   0   1   0   

## Steps 13–14: Query Vector and Relevance Scores

To search, I encode the query as a binary vector $q$ (same dimension as the keyword list), then compute:

$$d = q^T T$$

The result $d$ is a vector with one entry per page, where $d_j$ counts how many query keywords appear on page $j$. Pages with $d_j = 0$ are not relevant; pages with $d_j > 0$ are in the relevancy set.


In [12]:
query_keywords = ["Redwood", "Ash"]

q = np.zeros(len(all_keywords), dtype=int)
for keyword in query_keywords:
    if keyword in all_keywords:
        q[all_keywords.index(keyword)] = 1

d = q @ T

print(f"Search query: {query_keywords}\n")
print(f"{'Page':<6} {'Name':<25} {'Score':<8} {'Status'}")
print(f"{'-'*6} {'-'*25} {'-'*8} {'-'*20}")
for j, label in enumerate(pages):
    status = "RELEVANT" if d[j] > 0 else "not relevant"
    print(f"{label:<6} {page_names[label]:<25} {d[j]:<8} {status}")

relevant_pages = [pages[j] for j in range(num_pages) if d[j] > 0]
print(f"\nRelevancy set: {relevant_pages} ({len(relevant_pages)} of {num_pages} pages)")

Search query: ['Redwood', 'Ash']

Page   Name                      Score    Status
------ ------------------------- -------- --------------------
A      Trees For All             1        RELEVANT
B      The Weeping Willow        1        RELEVANT
C      Leaf Fall                 2        RELEVANT
D      Shady Place               0        not relevant
E      The Prickly Pine          1        RELEVANT
F      Good Wood                 2        RELEVANT
G      Lumber Jack Man           2        RELEVANT
H      Timber!                   2        RELEVANT
I      Fine Grains               0        not relevant
J      The Warped Board          1        RELEVANT
K      Evergreen Plants          1        RELEVANT
L      Sappy Planks              0        not relevant

Relevancy set: ['A', 'B', 'C', 'E', 'F', 'G', 'H', 'J', 'K'] (9 of 12 pages)


## Step 15: Final Search Results 

This is where everything comes together. I have two pieces of information about each page:
1. **Relevance** (from the term-document matrix): does the page match the query?
2. **Importance** (from PageRank): how structurally important is the page in the web graph?

Google's original insight: filter by relevance, then rank by importance. Among all pages that contain the search terms, show the most important ones first. Now it make so much sense why Google's results so much better than earlier search engines that ranked purely by keyword frequency.

In [13]:
results = []
for j in range(num_pages):
    if d[j] > 0:
        label = pages[j]
        results.append((label, pi_tilde_eigen[j], d[j]))

results.sort(key=lambda x: x[1], reverse=True)

print(f"Search results for {query_keywords}, ranked by PageRank (alpha={alpha}):\n")
print(f"{'Rank':<6} {'Page':<6} {'Name':<25} {'PageRank':<14} {'Matching Keywords'}")
print(f"{'-'*6} {'-'*6} {'-'*25} {'-'*14} {'-'*25}")
for rank, (label, pr, rel) in enumerate(results, 1):
    matching = [kw for kw in query_keywords if kw in page_keywords[label]]
    print(f"{rank:<6} {label:<6} {page_names[label]:<25} {pr:.8f}     {', '.join(matching)}")

Search results for ['Redwood', 'Ash'], ranked by PageRank (alpha=0.85):

Rank   Page   Name                      PageRank       Matching Keywords
------ ------ ------------------------- -------------- -------------------------
1      H      Timber!                   0.10780561     Redwood, Ash
2      C      Leaf Fall                 0.10655497     Redwood, Ash
3      B      The Weeping Willow        0.10374151     Redwood
4      G      Lumber Jack Man           0.10001115     Redwood, Ash
5      K      Evergreen Plants          0.08553085     Redwood
6      J      The Warped Board          0.08293668     Redwood
7      A      Trees For All             0.07642904     Ash
8      F      Good Wood                 0.07200759     Redwood, Ash
9      E      The Prickly Pine          0.02618392     Redwood
